In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import base64

import numpy as np
import scipy.optimize as opt
from itertools import product

https://docs.scipy.org/doc/scipy/reference/optimize.html

In [ ]:
# # PARMS
# changeable
org_id = 1

# fixed
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

In [ ]:

config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=org_id)

full_filepath_to_load = f"{path_to_local_data}{input_filename}"

In [ ]:
raw_eeg = pd.read_csv(f"{full_filepath_to_load}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")

In [ ]:

eeg_selected_feat = raw_eeg[["time", "mp_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del raw_eeg
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.describe())
work = eeg_selected_feat[
    (eeg_selected_feat["time"] >= pd.Timestamp(datetime(2025, 9, 23, 3, 0), tz='UTC')) &
    (eeg_selected_feat["time"] < pd.Timestamp(datetime(2025, 9, 23, 5, 15), tz='UTC'))
]
del eeg_selected_feat
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["mp_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_meas_gen", "wt_surp_gen":"sum_surp_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")
time_with_deficit = agg_on_time[agg_on_time["sum_surp_gen"] <= 0]
print(f"{len(time_with_deficit)}/{len(agg_on_time)} timestamps has surplus")


In [ ]:
work["org_id"] = 1
work

In [ ]:
work_tf_cons = pd.merge(left=work[work["energy_direction"] == 'C'], right=time_with_deficit, on="time", how="inner")
work_tf_cons

### Mengen

In [ ]:
EC = [org_id]
E = work[work["energy_direction"] == "G"]["mp_id"].unique()
V = work[work["energy_direction"] == "C"]["mp_id"].unique()
Z = np.append(V, E)
T = work["time"].unique()

m = {(z, ec): 1 for z in Z for ec in EC}

### Parameter

In [ ]:
groups = work.set_index(["mp_id", "time", "org_id"])["wt_meas_gen"]

idx = pd.MultiIndex.from_product([E, T, EC], names=["mp_id", "time", "org_id"])
values = groups.reindex(idx).tolist() 
keys = idx.tolist()                     

g = dict(zip(keys, values))

groups = work.set_index(["mp_id", "time", "org_id"])["wt_meas_cons"]

idx = pd.MultiIndex.from_product([V, T, EC], names=["mp_id", "time", "org_id"])
values = groups.reindex(idx).tolist()   
keys = idx.tolist()                     

c = dict(zip(keys, values))

### Varaiblen

In [ ]:
idx = {} 
pos=0




# 1) new variable z_t per time t, EC
L_max = 100 # len(V)
z_idx = {}
for t, ec in product(T, EC):
    for l in range(L_max):
        z_idx[('zL', (l, t, ec))] = pos
        pos += 1
n_z_levels = pos

# 2) x variables (consumers only)
for v, t, ec in product(V, T, EC):
    idx[('cc*', (v, t, ec))] = pos
    pos += 1
n_cc = pos - n_z_levels

# 3) y variables (producers only)
for e, t, ec in product(E, T, EC):
    idx[('s*', (e, t, ec))] = pos
    pos += 1
n_s = pos - n_cc  - n_z_levels

n_vars = pos                     # total number of decision variables
print(f"Total variables = {n_vars}  (cc*:{n_cc}, s*:{n_s}) n_z_levels : {n_z_levels}")

### Nebenbedingungen

#### Bounds

In [ ]:
bounds = []

for t, ec in product(T, EC):
    for l in range(L_max):
        bounds.append((0.0, None)) 

# x bounds (consumers only)
for v, t, ec in product(V, T, EC):
    low  = 0.0
    high = c[(v, t, ec)]
    bounds.append( (low, high) )

# y bounds (producers only)
for e, t, ec in product(E, T, EC):
    low  = 0.0
    high = g[(e, t, ec)]
    bounds.append( (low, high) )

#### Gleichheits Constraint

In [ ]:
A_ub = []
b_ub = []
A_eq = []
b_eq = []

In [ ]:
# old, with "pf"
# # (a) x‑definition
# for v, t, ec in product(V, T, EC):
#     row = np.zeros(n_vars)
#     row[idx[('cc*', (v, t, ec))]] = 100.0
#     row[idx[('pf', (v, t, ec))]] = -c[(v, t, ec)]
#     A_eq.append(row)
#     b_eq.append(0.0)

# # (b) y‑definition
# for e, t, ec in product(E, T, EC):
#     row = np.zeros(n_vars)
#     row[idx[('g*', (e, t, ec))]] = 100.0
#     row[idx[('pf', (e, t, ec))]] = -g[(e, t, ec)]
#     A_eq.append(row)
#     b_eq.append(0.0)

In [ ]:
# bounds of cc* and s* depending on c and g 
# -> i.e. that comm_cov cannot be more than the CMP consumed

A_ub = []
b_ub = []

for v, t, ec in product(V, T, EC):
    row = np.zeros(n_vars)
    row[idx[('cc*', (v, t, ec))]] = 1.0   # cc* coefficient
    A_ub.append(row)
    b_ub.append(c[(v, t, ec)])            # upper bound = c

for e, t, ec in product(E, T, EC):
    row = np.zeros(n_vars)
    row[idx[('s*', (e, t, ec))]] = 1
    A_eq.append(row)
    b_eq.append(g[(e, t, ec)])

In [ ]:
# Sum cc* <= sum g  (per time t)
for t, ec in product(T, EC):
    row = np.zeros(n_vars)

    # sum of the cc* (all consumers)
    for v in V:
        row[idx[('cc*', (v, t, ec))]] = 1.0

    # right-hand side = sum g(e,t,ec)
    rhs = sum(g[(e, t, ec)] for e in E)

    A_ub.append(row)
    b_ub.append(rhs)


In [ ]:
# ----------------------------
# z-Level-Kopplungen: cc* >= z_l
# ----------------------------
for t, ec in product(T, EC):
    for l in range(L_max):
        for v in V:
            row = np.zeros(n_vars)
            row[idx[('cc*', (v, t, ec))]] = -1.0
            row[z_idx[('zL', (l, t, ec))]] = 1.0
            A_ub.append(row)
            b_ub.append(0.0)  # -cc + z_l <= 0 => cc >= z_l


In [ ]:
# ----------------------------
# z-Level-Monotonie: z_1 <= z_2 <= z_3 ...
# ----------------------------
for t, ec in product(T, EC):
    for l in range(L_max - 1):
        row = np.zeros(n_vars)
        row[z_idx[('zL', (l, t, ec))]] = 1.0
        row[z_idx[('zL', (l + 1, t, ec))]] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)  # z_l - z_{l+1} <= 0

In [ ]:
# T_S = [t for t in T if sum(g[(e, t, org_id)] for e in E) >= sum(c[(v, t, org_id)] for v in V)]
# T_U = [t for t in T if t not in T_S]       # komplement


### Zielfunktion

In [ ]:
# # ----- 6) objective function -----
# # optional: maximise total cc* (distribute all water)
# c_obj = np.zeros(n_vars)
# for v, t, ec in product(V, T, EC):
#     c_obj[idx[('cc*', (v, t, ec))]] = -1.0  # linprog minimises, so -1 → max

In [ ]:
# ----------------------------
# objective function: max z_L (top level) for all times
# ----------------------------
c_obj = np.zeros(n_vars)
for t, ec in product(T, EC):
    z_top_idx = z_idx[('zL', (L_max-1, t, ec))]
    c_obj[z_top_idx] = -1.0  # linprog minimiert, also -1 -> max

### Summary of the matrices

In [ ]:
res = opt.linprog(
    c=c_obj,                # objective-function coefficients
    A_eq=A_eq, b_eq=b_eq,  # equality constraints
    A_ub=A_ub, b_ub=b_ub,  # inequalities (if any)
    bounds=bounds,
    method='highs',         # SciPy's current, very robust solver
    options={'disp': True} # print iteration info
)

print("\n=== Solver result ===")
if res.success:
    print("Optimal objective value (negative = covered demand):", res.fun)
    # -----------------------------------------------------------------
    #   back-transformation into the three variable sets
    # -----------------------------------------------------------------
    cc_opt  = { (v, t, ec): res.x[idx[('cc*', (v, t, ec))]]
               for v, t, ec in product(V, T, EC) }

    s_opt  = { (e, t, ec): res.x[idx[('s*', (e, t, ec))]]
               for e, t, ec in product(E, T, EC) }

    # -------------------------------------------------------------
    #   example output (small demo set only)
    # -------------------------------------------------------------

    print("\nAllocated REC energy for the consumers (c*) in kWh:")
    for v, t, ec in product(V, T, EC):
        print(f"  c*[{v},{t}] = {cc_opt[(v,t,ec)]:.2f}")

    print("\nGeneration amount after control (g*) in kWh:")
    for e, t, ec in product(E, T, EC):
        print(f"  g*[{e},{t}] = {s_opt[(e,t,ec)]:.2f}")

else:
    print("LP solution failed:", res.message)

In [ ]:
cc_opt_output = pd.DataFrame(
    [(k[0], k[1], k[2], v) for k, v in cc_opt.items()],
    columns=["mp_id", "time", "org_id", "cc*"],
)

tf_schedule = pd.merge(
    left=work[["mp_id", "time", "org_id", "wt_meas_cons"]],
    right=cc_opt_output,
    on=["mp_id","time", "org_id"],
    how="left",
)

tf_schedule["pf"] = tf_schedule["cc*"] /  tf_schedule["wt_meas_cons"] * 100

tf_schedule = tf_schedule[["time", "mp_id", "pf"]].copy()
tf_schedule["pf"] = tf_schedule["pf"].fillna(100).clip(upper=100)

In [ ]:
def apply_pf_schedule_to_mps(all_mps: pd.DataFrame, pf_schedule: pd.DataFrame):
    def gini(x):
        total = 0
        for i, xi in enumerate(x[:-1], 1):
            total += np.sum(np.abs(xi - x[i:]))
        return total / (len(x)**2 * np.mean(x))
    # TODO NOTE!!! this only works on PF for C-MPS
    # TODO NOTE!!! this only works on PF during "Deficit", i dont how the calculations apply/what they do, during surplus   
    
    applied_pfs = pd.merge(left = all_mps, right=pf_schedule, on=["time", "mp_id"], how="outer").merge(right=agg_on_time, on="time", how="left")
    applied_pfs["pf"] = applied_pfs["pf"].fillna(100)
    applied_pfs["pf"] /= 100
    applied_pfs["surp_ratio"] = (applied_pfs["sum_meas_gen"] / applied_pfs["sum_meas_cons"]).replace(np.nan, 1).clip(upper=1) 

    print(f"Original Sums:")
    print(f"\t wt_meas_cons (c): {applied_pfs["wt_meas_cons"].sum():.3f}")
    print(f"\t comm_cov (cc): {applied_pfs["comm_cov"].sum():.3f}")
    old_restnetzbezug = (applied_pfs["wt_meas_cons"].sum() - applied_pfs["comm_cov"].sum())
    print(f"\t -> Restnetzbezug (c - cc): {old_restnetzbezug:.3f}")
    print(f"\t comm_pot: {applied_pfs["comm_pot"].sum():.3f}")
    print(f"\t wt_meas_gen: {applied_pfs["wt_meas_gen"].sum():.3f}")
    print(f"\t wt_surp_gen: {applied_pfs["wt_surp_gen"].sum():.3f}")

    applied_pfs["opt_meas_cons"] = applied_pfs["wt_meas_cons"] * applied_pfs["pf"]
    applied_pfs["opt_comm_cov"] = applied_pfs["comm_cov"] * applied_pfs["pf"]
    applied_pfs["opt_comm_pot"] = applied_pfs["comm_pot"] * applied_pfs["pf"]

    new_calced_agg_on_time = applied_pfs.groupby(by="time").sum().reset_index()[["time", "opt_meas_cons", "opt_comm_cov", "opt_comm_pot"]].rename(columns={"opt_comm_cov":"sum_opt_comm_cov", "opt_comm_pot":"sum_opt_comm_pot", "opt_meas_cons":"sum_opt_meas_cons"})
    applied_pfs = applied_pfs.merge(new_calced_agg_on_time, on="time", how="left")

    applied_pfs["opt_surp_ratio"] = (applied_pfs["sum_meas_gen"] / applied_pfs["sum_opt_meas_cons"]).replace(np.nan, 1).clip(upper=1) 

    applied_pfs["opt_comm_pot"] = applied_pfs["opt_surp_ratio"] * applied_pfs["opt_meas_cons"]
    applied_pfs["opt_comm_cov"] = applied_pfs["opt_surp_ratio"] * applied_pfs["opt_meas_cons"]

    print(f"\nOptimized Sums:")
    print(f"\t opt_meas_cons: {applied_pfs["opt_meas_cons"].sum():.3f} [wt_meas_cons: {applied_pfs["wt_meas_cons"].sum():.3f}, reduction allowed]")
    print(f"\t opt_comm_cov: {applied_pfs["opt_comm_cov"].sum():.3f} [comm_cov: {applied_pfs["comm_cov"].sum():.3f}, reduction NOT! allowed]")
    opt_restnetzbezug = (applied_pfs["opt_meas_cons"].sum() - applied_pfs["comm_cov"].sum())
    print(f"\t -> opt_Restnetzbezug (c* - cc*): {opt_restnetzbezug:.3f}")
    print(f"\t -> real Restnetzbezug: {old_restnetzbezug}")
    print(f"\t -> real Restnetzbezug + opt_comm_cov = wt_meas_cons / {old_restnetzbezug:.3f} + {applied_pfs["opt_comm_cov"].sum():.3f} = {applied_pfs["wt_meas_cons"].sum():.3f}")
    print(f"\t opt_comm_pot: {applied_pfs["opt_comm_pot"].sum():.3f} [comm_pot: {applied_pfs["comm_pot"].sum():.3f}, reduction NOT! allowed]")

    print(f"\t wt_meas_gen: {applied_pfs["wt_meas_gen"].sum():.3f}")
    print(f"\t wt_surp_gen: {applied_pfs["wt_surp_gen"].sum():.3f}")

    sums_on_c_mps = applied_pfs[applied_pfs["energy_direction"] == "C"].groupby(by="mp_id").sum(numeric_only=True)  
    print(f"\nGini on Original cc: {gini(sums_on_c_mps["comm_cov"]):.3f}")
    print(f"Gini on optimized cc: {gini(sums_on_c_mps["opt_comm_cov"]):.3f} [should be lower than original]")

    return applied_pfs.copy()


In [ ]:
applied_pfs = apply_pf_schedule_to_mps(work, tf_schedule)

In [ ]:
def plot_stacked_gain_loss_sortable(df, col1, col2):
    df = df.reset_index()

    # helper function: compute data for one sort order
    def compute_sorted(sort_col):
        df_sorted = df.sort_values(sort_col, ascending=False)

        x = np.arange(1, len(df_sorted) + 1)
        base = df_sorted[col1]
        compare = df_sorted[col2]

        diff = compare - base
        positive_diff = diff.clip(lower=0)
        negative_diff = (-diff).clip(lower=0)
        common = np.minimum(base, compare)

        return {
            "x": x,
            "common": common,
            "neg": negative_diff,
            "pos": positive_diff,
            "mp_id": df_sorted["mp_id"]
        }

    # precompute for each sort order
    data_col1 = compute_sorted(col1)
    data_col2 = compute_sorted(col2)

    # ─────────────────────────────────────────
    # 1) create figure with the first (col1) data
    # ─────────────────────────────────────────
    fig = go.Figure()

    # blue
    fig.add_bar(
        x=data_col1["x"],
        y=data_col1["common"],
        marker_color="blue",
        name="Base",
        hovertext=data_col1["mp_id"],     # MP-ID in hover
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
    )
    # red
    fig.add_bar(
        x=data_col1["x"],
        y=data_col1["neg"],
        marker_color="red",
        name=f"Reductions: {col1} - {col2}",
        hovertext=data_col1["mp_id"],     # MP-ID in hover
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
    )
    # green
    fig.add_bar(
        x=data_col1["x"],
        y=data_col1["pos"],
        marker_color="green",
        name=f"Gains: {col2} - {col1}",
        hovertext=data_col1["mp_id"],     # MP-ID in hover
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
    )

    # ─────────────────────────────────────────
    # 2) buttons that update the EXISTING traces
    # ─────────────────────────────────────────
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                showactive=True,
                y=-0.12,
                x=0.5,
                xanchor="center",
                # yanchor="bottom",
                direction="right",
                font=dict(size=9),
                pad=dict(l=0, r=0, t=0, b=0),
                buttons=[
                    dict(
                        label=f"sorted by {col1}",
                        method="update",
                        args=[
                            {
                                "x": [
                                    data_col1["x"],  # trace 0
                                    data_col1["x"],  # trace 1
                                    data_col1["x"],  # trace 2
                                ],
                                "y": [
                                    data_col1["common"],
                                    data_col1["neg"],
                                    data_col1["pos"],
                                ],
                                "hovertext": [
                                    data_col1["mp_id"],
                                    data_col1["mp_id"],
                                    data_col1["mp_id"]
                                ]
                            },
                            {"title": f"Participation Factor Opt Results (sorted by {col1})"}
                        ]
                    ),
                    dict(
                        label=f"sorted by {col2}",
                        method="update",
                        args=[
                            {
                                "x": [
                                    data_col2["x"],
                                    data_col2["x"],
                                    data_col2["x"],
                                ],
                                "y": [
                                    data_col2["common"],
                                    data_col2["neg"],
                                    data_col2["pos"],
                                ],
                                "hovertext": [
                                    data_col2["mp_id"],
                                    data_col2["mp_id"],
                                    data_col2["mp_id"]
                                ]
                            },
                            {"title": f"Participation Factor Opt Results (sorted by {col2})"}
                        ]
                    )
                ]
            )
        ]
    )

    fig.update_layout(
        barmode="stack",
        title=f"Participation Factor Opt Results (sorted by {col1})",
        xaxis_title="sorted Order",
        yaxis_title=f"kWh",
        template="plotly_white",
        legend=dict(
            orientation="h",
            x=0,
            y=1.1,
            xanchor="left",
            yanchor="top",
        ),
        xaxis_title_standoff=5,
        margin=dict(t=85, b=10)
    )

    fig.show()


In [ ]:
single_time_filtered = applied_pfs

check_full_calc = single_time_filtered[single_time_filtered["energy_direction"]=="C"].groupby(by="mp_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov"]]
check_full_calc["cc_diff"] = check_full_calc["comm_cov"] - check_full_calc["opt_comm_cov"]
check_full_calc.sort_values(by="cc_diff", ascending=False)

plot_stacked_gain_loss_sortable(check_full_calc, "comm_cov", "opt_comm_cov")

In [ ]:
sns.histplot(tf_schedule["pf"])